In [6]:
library(tidyverse)
library(bigrquery)

# This query represents dataset "fitbit_no_condition" for domain "fitbit_activity" and was generated for All of Us Controlled Tier Dataset v8
dataset_81383123_fitbit_activity_sql <- paste("
    SELECT
        activity_summary.person_id,
        activity_summary.date,
        activity_summary.activity_calories,
        activity_summary.calories_bmr,
        activity_summary.calories_out,
        activity_summary.elevation,
        activity_summary.fairly_active_minutes,
        activity_summary.floors,
        activity_summary.lightly_active_minutes,
        activity_summary.marginal_calories,
        activity_summary.sedentary_minutes,
        activity_summary.steps,
        activity_summary.very_active_minutes 
    FROM
        `activity_summary` activity_summary   
    WHERE
        activity_summary.PERSON_ID IN (SELECT
            distinct person_id  
        FROM
            `cb_search_person` cb_search_person  
        WHERE
            cb_search_person.person_id IN (SELECT
                person_id 
            FROM
                `cb_search_person` p 
            WHERE
                has_fitbit = 1 ) 
            AND cb_search_person.person_id IN (SELECT
                person_id 
            FROM
                `cb_search_person` p 
            WHERE
                has_fitbit_activity_summary = 1 ) 
            AND cb_search_person.person_id IN (SELECT
                person_id 
            FROM
                `cb_search_person` p 
            WHERE
                has_fitbit_plus_device_information = 1 ) 
            AND cb_search_person.person_id IN (SELECT
                person_id 
            FROM
                `cb_search_person` p 
            WHERE
                has_fitbit_heart_rate_summary = 1 ) 
            AND cb_search_person.person_id IN (SELECT
                person_id 
            FROM
                `cb_search_person` p 
            WHERE
                has_fitbit_heart_rate_level = 1 ) 
            AND cb_search_person.person_id IN (SELECT
                person_id 
            FROM
                `cb_search_person` p 
            WHERE
                has_fitbit_steps_intraday = 1 ) 
            AND cb_search_person.person_id IN (SELECT
                person_id 
            FROM
                `cb_search_person` p 
            WHERE
                has_fitbit_sleep_daily_summary = 1 ) 
            AND cb_search_person.person_id IN (SELECT
                person_id 
            FROM
                `cb_search_person` p 
            WHERE
                has_fitbit_sleep_level = 1 ) 
            AND cb_search_person.person_id NOT IN (SELECT
                criteria.person_id 
            FROM
                (SELECT
                    DISTINCT person_id, entry_date, concept_id 
                FROM
                    `cb_search_all_events` 
                WHERE
                    (concept_id IN(SELECT
                        DISTINCT c.concept_id 
                    FROM
                        `cb_criteria` c 
                    JOIN
                        (SELECT
                            CAST(cr.id as string) AS id       
                        FROM
                            `cb_criteria` cr       
                        WHERE
                            concept_id IN (4274025)       
                            AND full_text LIKE '%_rank1]%'      ) a 
                            ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                            OR c.path LIKE CONCAT('%.', a.id) 
                            OR c.path LIKE CONCAT(a.id, '.%') 
                            OR c.path = a.id) 
                    WHERE
                        is_standard = 1 
                        AND is_selectable = 1) 
                    AND is_standard = 1 )) criteria ) 
            AND cb_search_person.person_id NOT IN (SELECT
                criteria.person_id 
            FROM
                (SELECT
                    DISTINCT person_id, entry_date, concept_id 
                FROM
                    `cb_search_all_events` 
                WHERE
                    (concept_id IN(SELECT
                        DISTINCT c.concept_id 
                    FROM
                        `cb_criteria` c 
                    JOIN
                        (SELECT
                            CAST(cr.id as string) AS id       
                        FROM
                            `cb_criteria` cr       
                        WHERE
                            concept_id IN (135930, 4101796, 4117930, 255919, 444112, 432453, 4024567, 4274025, 376106, 4011630, 4213101, 4038502, 4093991, 40480457, 4041284, 4042141, 43021226, 4041283, 4227253, 4023995, 4102111, 138239, 4293175, 4212577, 4302537, 4021667)       
                            AND full_text LIKE '%_rank1]%'      ) a 
                            ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                            OR c.path LIKE CONCAT('%.', a.id) 
                            OR c.path LIKE CONCAT(a.id, '.%') 
                            OR c.path = a.id) 
                    WHERE
                        is_standard = 1 
                        AND is_selectable = 1) 
                    AND is_standard = 1 )) criteria ) )", sep="")

# Formulate a Cloud Storage destination path for the data exported from BigQuery.
# NOTE: By default data exported multiple times on the same day will overwrite older copies.
#       But data exported on a different days will write to a new location so that historical
#       copies can be kept as the dataset definition is changed.
fitbit_activity_81383123_path <- file.path(
  Sys.getenv("WORKSPACE_BUCKET"),
  "bq_exports",
  Sys.getenv("OWNER_EMAIL"),
  strftime(lubridate::now(), "%Y%m%d"),  # Comment out this line if you want the export to always overwrite.
  "fitbit_activity_81383123",
  "fitbit_activity_81383123_*.csv")
message(str_glue('The data will be written to {fitbit_activity_81383123_path}. Use this path when reading ',
                 'the data into your notebooks in the future.'))

# Perform the query and export the dataset to Cloud Storage as CSV files.
# NOTE: You only need to run `bq_table_save` once. After that, you can
#       just read data from the CSVs in Cloud Storage.
bq_table_save(
  bq_dataset_query(Sys.getenv("WORKSPACE_CDR"), dataset_81383123_fitbit_activity_sql, billing = Sys.getenv("GOOGLE_PROJECT")),
  fitbit_activity_81383123_path,
  destination_format = "CSV")


The data will be written to gs://fc-secure-1d0240db-6bfa-4a11-a526-48cfbb0d664a/bq_exports/zqiu@researchallofus.org/20260421/fitbit_activity_81383123/fitbit_activity_81383123_*.csv. Use this path when reading the data into your notebooks in the future.



In [7]:
# Read the data directly from Cloud Storage into memory.
# NOTE: Alternatively you can `gsutil -m cp {fitbit_activity_81383123_path}` to copy these files
#       to the Jupyter disk.
read_bq_export_from_workspace_bucket <- function(export_path) {
  col_types <- NULL
  bind_rows(
    map(system2('gsutil', args = c('ls', export_path), stdout = TRUE, stderr = TRUE),
        function(csv) {
          message(str_glue('Loading {csv}.'))
          chunk <- read_csv(pipe(str_glue('gsutil cat {csv}')), col_types = col_types, show_col_types = FALSE)
          if (is.null(col_types)) {
            col_types <- spec(chunk)
          }
          chunk
        }))
}
healthy_activity <- read_bq_export_from_workspace_bucket(fitbit_activity_81383123_path)

dim(healthy_activity)

head(healthy_activity, 5)

Loading gs://fc-secure-1d0240db-6bfa-4a11-a526-48cfbb0d664a/bq_exports/zqiu@researchallofus.org/20260421/fitbit_activity_81383123/fitbit_activity_81383123_000000000000.csv.

Loading gs://fc-secure-1d0240db-6bfa-4a11-a526-48cfbb0d664a/bq_exports/zqiu@researchallofus.org/20260421/fitbit_activity_81383123/fitbit_activity_81383123_000000000001.csv.

Loading gs://fc-secure-1d0240db-6bfa-4a11-a526-48cfbb0d664a/bq_exports/zqiu@researchallofus.org/20260421/fitbit_activity_81383123/fitbit_activity_81383123_000000000002.csv.

Loading gs://fc-secure-1d0240db-6bfa-4a11-a526-48cfbb0d664a/bq_exports/zqiu@researchallofus.org/20260421/fitbit_activity_81383123/fitbit_activity_81383123_000000000003.csv.

Loading gs://fc-secure-1d0240db-6bfa-4a11-a526-48cfbb0d664a/bq_exports/zqiu@researchallofus.org/20260421/fitbit_activity_81383123/fitbit_activity_81383123_000000000004.csv.

Loading gs://fc-secure-1d0240db-6bfa-4a11-a526-48cfbb0d664a/bq_exports/zqiu@researchallofus.org/20260421/fitbit_activity_81383123/

[1] 18364745       13

person_id,date,activity_calories,calories_bmr,calories_out,elevation,fairly_active_minutes,floors,lightly_active_minutes,marginal_calories,sedentary_minutes,steps,very_active_minutes
<dbl>,<date>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
6349547,2020-04-03,1099,1602,2536,120,2,12,115,722,1115,12195,83
6349547,2020-09-22,1542,1603,2954,120,12,12,184,1057,1185,7804,59
6349547,2022-02-03,1356,1591,2750,120,29,12,186,851,706,11771,55
6350258,2017-01-12,959,1571,2385,120,10,12,251,448,713,2647,0
6350258,2015-12-23,935,1558,2306,120,60,12,150,514,859,3952,12


In [8]:
source('/home/jupyter/workspaces/temporaldigitalhealthphenotyping/ind_descriptive_func.R')

In [9]:
wear_compliance_func(path.out = '/home/jupyter/workspaces/result/Healthy/', df = healthy_activity)


── Wear compliance ──
     n_days       followup_days 
 Min.   :   1.0   Min.   :   1  
 1st Qu.: 108.0   1st Qu.: 117  
 Median : 534.5   Median : 708  
 Mean   : 799.5   Mean   :1071  
 3rd Qu.:1154.0   3rd Qu.:1741  
 Max.   :5386.0   Max.   :5386  


`geom_smooth()` using formula = 'y ~ x'


   person_id           n_days         first_date           last_date         
 Min.   :1000107   Min.   :   1.0   Min.   :2008-12-31   Min.   :2015-07-12  
 1st Qu.:2099866   1st Qu.: 108.0   1st Qu.:2017-12-31   1st Qu.:2023-08-17  
 Median :3769178   Median : 534.5   Median :2021-07-22   Median :2023-10-01  
 Mean   :4541989   Mean   : 799.5   Mean   :2020-06-06   Mean   :2023-05-11  
 3rd Qu.:6864914   3rd Qu.:1154.0   3rd Qu.:2023-06-03   3rd Qu.:2023-10-01  
 Max.   :9999996   Max.   :5386.0   Max.   :2023-09-30   Max.   :2023-10-01  
 followup_days   missing_days     pct_missing              compliance_tier 
 Min.   :   1   Min.   :   0.0   Min.   : 0.000   High (>90%)      :14028  
 1st Qu.: 117   1st Qu.:   0.0   1st Qu.: 0.000   Moderate (70-90%): 3817  
 Median : 708   Median :  16.0   Median : 3.846   Low (50-70%)     : 2167  
 Mean   :1071   Mean   : 271.4   Mean   :16.966   Poor (<50%)      : 2958  
 3rd Qu.:1741   3rd Qu.: 222.0   3rd Qu.:25.774                           

NULL

In [10]:
patient_stats <- activity_prep_stats(path.out = '/home/jupyter/workspaces/Data/', cohort = 'Healthy', df = healthy_activity)

# A tibble: 6 × 9
  metric                     N   Mean     SD Median     Q1     Q3      P5    P95
  <chr>                  <int>  <dbl>  <dbl>  <dbl>  <dbl>  <dbl>   <dbl>  <dbl>
1 calories_out           22970 2243.  5.71e2 2.16e3 1.83e3 2586.  1.46e+3 3.28e3
2 fairly_active_minutes  22969   15.4 1.47e1 1.16e1 5.47e0   20.9 8.45e-1 4.28e1
3 lightly_active_minutes 22970  179.  7.95e1 1.81e2 1.29e2  230.  3.71e+1 3.06e2
4 sedentary_minutes      22970  919.  2.26e2 8.73e2 7.47e2 1084.  6.15e+2 1.34e3
5 steps                  22970 6591.  2.81e4 6.06e3 3.79e3 8494.  9.42e+2 1.28e4
6 very_active_minutes    22969   15.9 1.83e1 9.85e0 3.52e0   22.0 3.75e-1 5.07e1


In [ ]:
 eligible_ids <- filter_pid_func(path.out = '/home/jupyter/workspaces/Data/',
    cohort = 'Healthy', df = healthy_activity, type = 'activity')
